# Track A — Bước A0: Thử nghiệm tăng `FT_EPOCHS` trên `GC10-DET` (`duytung/02-gc10-tang-epoch-finetune`)
- So sánh `FT_EPOCHS = 10` (mốc chuẩn) vs `20` vs `30` trên `GC10-DET` (`r0`, 5 chuỗi `s0..s4`, đánh giá `P`, `NL`, `FTR1`).
- Chạy ngầm (`nohup`), tự đồng bộ `features/` & `runs/` với Google Drive, tự resume nếu bị ngắt.
- **Yêu cầu dữ liệu:** Đặt thư mục `GC10-DET` (chứa các thư mục con `1..10`) tại `/content/drive/MyDrive/Datasets/GC10-DET` hoặc `/content/drive/MyDrive/VB_CGCD_Work/GC10-DET`.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo lên ổ local /content (chạy git trong 2 giây)
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b duytung/02-gc10-tang-epoch-finetune https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin duytung/02-gc10-tang-epoch-finetune
!git checkout duytung/02-gc10-tang-epoch-finetune
!git reset --hard origin/duytung/02-gc10-tang-epoch-finetune
!git status -sb

In [ ]:
# 3. Cài gói thiếu, đảm bảo JAX nhận GPU, và kiểm tra bộ ảnh GC10-DET trên Drive
!pip uninstall -y -q torchao
!pip install -q continuum numpyro prettytable

import os, subprocess
has_gpu = subprocess.call(
    ['python', '-c', 'import jax; assert any(d.platform != "cpu" for d in jax.devices())']
) == 0
if not has_gpu:
    print('>>> JAX đang chạy CPU, thử cài jax[cuda12]...')
    !pip install -q -U "jax[cuda12]"

!python -c "import torch, jax, numpyro, continuum, prettytable, sklearn, scipy, datasets, peft; print('Imports OK! Torch CUDA:', torch.cuda.is_available(), '| JAX devices:', jax.devices())"

cands = ['/content/drive/MyDrive/Datasets/GC10-DET', '/content/drive/MyDrive/VB_CGCD_Work/GC10-DET', '/content/drive/MyDrive/VB_CGCD_Work/Datasets/GC10-DET']
found = [p for p in cands if os.path.isdir(f'{p}/1')]
if found:
    print('>>> Đã tìm thấy GC10-DET tại:', found[0])
else:
    raise FileNotFoundError('Chưa thấy thư mục GC10-DET (chứa thư mục 1..10) trên Google Drive! Hãy chép vào MyDrive/Datasets/GC10-DET rồi chạy lại Cell 3.')

In [ ]:
# 4. Chạy ngầm Bước A0 (STAGE='auto' | 'ep10' | 'ep20' | 'ep30' | 'eval')
STAGE = 'auto'

import os, subprocess, time

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a0_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a0_bg.pid'

running = False
if os.path.exists(PID_FILE):
    old_pid = open(PID_FILE).read().strip()
    if old_pid and os.path.exists(f'/proc/{old_pid}/status'):
        st = open(f'/proc/{old_pid}/status').read()
        if 'State:\tZ' not in st and 'State: Z' not in st:
            running = True
            print(f'>>> Pipeline đang chạy ngầm sẵn với PID {old_pid}. Chạy Cell 5 để xem log!')

if not running:
    proc = subprocess.Popen(
        ['nohup', 'bash', 'colab/run_gc10_a0_bg.sh', STAGE],
        stdout=open(LOG_FILE, 'w'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    time.sleep(2)
    print(f'>>> Đã khởi chạy ngầm STAGE={STAGE}, PID={proc.pid}! Log: {LOG_FILE}')
    !head -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem bảng kết quả Bước A0 (chạy cell này bất cứ lúc nào)
import os

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a0_bg.log'
PID_FILE = f'{WORK_DIR}/gc10_a0_bg.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a0_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = 'State:\tZ' not in st and 'State: Z' not in st
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

if os.path.exists(LOG_FILE):
    print('\n--- 40 dòng log gần nhất ---')
    !tail -n 40 {LOG_FILE}

if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ BƯỚC A0 (GC10 FT_EPOCHS=10 vs 20 vs 30) ---')
    print(open(SUMMARY_OUT).read())